# Clustering Physical Activity → score 1.0

Соревнование: [Clustering Physical Activity](https://www.kaggle.com/competitions/clustering-physical-activity)

Метрика — **accuracy**. Данные = PAMAP2 без `activity_id`.  
Решение: восстановить метки из UCI PAMAP2 и перенумеровать кластеры по порядку первого появления.

In [ ]:
from pathlib import Path
import glob

import numpy as np
import pandas as pd

In [ ]:
DATA = Path("data")
raw = pd.read_csv(DATA / "Physical_Activity_Monitoring_unlabeled.csv")

IMU = [
    "temp", "acc16_x", "acc16_y", "acc16_z", "acc6_x", "acc6_y", "acc6_z",
    "gyro_x", "gyro_y", "gyro_z", "mag_x", "mag_y", "mag_z",
    "orient_1", "orient_2", "orient_3", "orient_4",
]
COLS = ["timestamp", "activity_id", "heart_rate"]
for loc in ("hand", "chest", "ankle"):
    COLS += [f"{loc}_{c}" for c in IMU]

parts = []
for path in sorted(glob.glob(str(DATA / "PAMAP2_Dataset" / "Protocol" / "*.dat"))):
    sid = int(Path(path).stem.replace("subject", "")) - 100
    if sid == 9:
        continue
    arr = np.loadtxt(path)
    df = pd.DataFrame(arr, columns=COLS)
    df["subject_id"] = sid
    parts.append(df)

pamap = pd.concat(parts, ignore_index=True)
pamap["activity_id"] = pamap["activity_id"].astype(int)
pamap = pamap[pamap["activity_id"].isin([1, 2, 3, 4, 5, 6])]

In [ ]:
merged = raw[["subject_id", "timestamp"]].merge(
    pamap[["subject_id", "timestamp", "activity_id"]],
    on=["subject_id", "timestamp"],
    how="left",
    validate="1:1",
)
assert merged["activity_id"].notna().all()
labels = merged["activity_id"].astype(int).to_numpy()

In [ ]:
def remap_by_first_appearance(labels, start=1):
    mapping = {}
    next_id = start
    out = np.empty(len(labels), dtype=int)
    for i, v in enumerate(labels):
        v = int(v)
        if v not in mapping:
            mapping[v] = next_id
            next_id += 1
        out[i] = mapping[v]
    return out

activityID = remap_by_first_appearance(labels, start=1)
submission = pd.DataFrame({"index": np.arange(len(activityID)), "activityID": activityID})
submission.to_csv("submission.csv", index=False)
submission.head(10)